# Sim 2: Tracking a trajectory under feedback, with velocity inputs

**Roles today.** The **Driver** says the next step out loud. The **Keyboard**
types only what someone else has said. The **Checker** owns the board.

**The rule.** Every cell that moves the robot is preceded by a 🔮 PREDICT. Before
anyone runs it, the Checker writes the group's prediction on the board. After it
runs, the group says what matched, what did not, and why, before moving on.
Run every cell, in order, with Shift+Enter.

**The clock.** Part 0 by 0:10, Part A by 0:28, then screens up. B1 by 0:48, B2 by
0:62, then Part C together. If you are behind, skip to Part C when the instructor
calls it; the submission finishes what you started.

**Today's robot.** The capstone cell: the UR5 with its suction cup, over the
table, in **velocity mode**. Every time step (0.05 s) your code sends six joint
rates in rad/s, and the joint motors hold those rates until the next step. That
is the setting of MR 11.3: the controller's output is a velocity, not a torque.
It is also how the capstone's controller drives the arm.

**Never leave a rate running.** Between cells, CoppeliaSim runs on its own. A
joint left at 0.3 rad/s keeps turning while you read. Every helper below ends by
commanding zero; if you write your own loop, end it the same way.

In [ ]:
# the next line makes figures interactive: drag to zoom, instead of static pictures
%matplotlib widget
import numpy as np                 # arrays and linear algebra
import matplotlib.pyplot as plt    # plots of the error against time
import screws                      # the course's robotics library
from me465 import open_capstone, close
from me465.capstone import T_SW    # takes CoppeliaSim's world frame to {s}, as in Sim 1

np.set_printoptions(precision=4, suppress=True)

# The model: the scene's UR5 with its cup, in {s}, read off the scene the way Sim 1 read the
# screw axes. Kinematic mode, so nothing moves while we read.
scene, arm, cell = open_capstone("kinematic")
world = arm.robot()                                    # M at the flange, and the screw axes, in the world frame
T_flange_cup = screws.transform_inv(T_SW @ arm.tip_frame()) @ cell.tool_frame()   # the cup on the flange
M = T_SW @ world.M @ T_flange_cup                      # the cup's face at the zero pose, in {s}
S = screws.adjoint(T_SW) @ world.S                     # the six screw axes, in {s}
robot = screws.Robot.from_screw_axes(M, list(S.T), name="UR5 + cup")
close(scene)
scene = None

# Two poses of the cup, both pointing straight down, 0.30 m above the table.
DOWN = np.array([[1, 0, 0], [0, -1, 0], [0, 0, -1]])
X_A = screws.rp_to_transform(DOWN, [0.35, -0.20, 0.30])
X_B = screws.rp_to_transform(screws.rot([0, 0, 1], np.radians(45)) @ DOWN, [0.35, 0.20, 0.30])
ELBOW_UP = np.radians([-126.9, 14.5, 120.2, -44.7, -90, -51.9])   # a seed that keeps the elbow above the table
theta_A = robot.ik(X_A, ELBOW_UP).theta                           # the arm at A


def start_at(theta):
    # (Re)load the cell with the arm at the joint angles theta, in velocity mode, holding still.
    global scene, arm, cell
    if scene is not None:
        close(scene)
    scene, arm, cell = open_capstone("velocity", theta)


start_at(theta_A)
dt = scene.dt
print("time step:", dt, "s;  the cup at A, in {s}:\n", cell.tool_frame())

## Part 0: Drive with velocities (warm-up)

`drive(rates, seconds)` holds six joint rates (rad/s) for a while, then commands
zero. Joint 1 turns the whole arm about the vertical axis through the base.

🔮 **PREDICT.** With joint 1 at 0.3 rad/s for 2 s: how far does it turn, in
degrees? Where does the cup end up? Then edit the rates and run it again: try a
negative rate, and a different joint. Small rates; the table is 0.3 m below the cup.

In [ ]:
def drive(rates, seconds):
    # Hold joint rates (rad/s) for `seconds`, then zero. Returns the angles before and after.
    arm.command_velocities(np.zeros(6)); scene.step()      # take the clock first, holding still
    before = arm.theta()
    for _ in range(int(round(seconds / dt))):
        arm.command_velocities(rates)                       # the rates, held for one step
        scene.step()
    arm.command_velocities(np.zeros(6)); scene.step()       # never leave a rate running
    scene.release()                                         # hand the clock back between cells
    return before, arm.theta()


rates = [0.3, 0, 0, 0, 0, 0]          # rad/s; edit me and re-run
before, after = drive(rates, 2.0)
print("turned (deg):", np.round(np.degrees(after - before), 3))

🔮 **PREDICT.** Now command zero for 5 s. Does anything move? If so, which
joints, and why those?

In [ ]:
before, after = drive(np.zeros(6), 5.0)
print("moved while commanded zero (deg):", np.round(np.degrees(after - before), 4))

## Part A: One joint (MR 11.3.1)

Joint 1 alone, with the other five held still. Joint 1 turns about the vertical,
so gravity does not load it. Its reference starts **0.2 rad ahead** of where the
joint is and moves at a constant **c = 0.3 rad/s** for 4 s:

$$\theta_d(t) = \theta(0) + 0.2 + c\,t, \qquad \theta_e(t) = \theta_d(t) - \theta(t).$$

### A1. The control law

Write MR's feedforward-plus-PI law (eq. 11.15) for one joint in `joint_rate`:
the commanded rate is the reference's rate $\dot\theta_d$ (only if `ff` is on),
plus $K_p \theta_e$, plus $K_i$ times the integral of the error. `track_joint` runs
it and returns the error at every step.

In [ ]:
def joint_rate(e, integral, thetadot_d, Kp, Ki, ff):
    # MR eq. 11.15 for one joint: the commanded rate, in rad/s.
    return 0.0   # TODO: thetadot_d (only if ff), plus Kp * e, plus Ki * integral


def track_joint(Kp, Ki=0.0, ff=True, c=0.3, e0=0.2, seconds=4.0):
    # Run joint 1 along its ramp under joint_rate; return (t, error, peak commanded rate).
    start_at(theta_A)
    arm.command_velocities(np.zeros(6)); scene.step()
    theta0 = arm.theta()
    integral, t, err, peak = 0.0, [], [], 0.0
    for k in range(int(round(seconds / dt)) + 1):
        theta = arm.theta()
        e = theta0[0] + e0 + c * k * dt - theta[0]
        integral += e * dt
        rates = np.zeros(6)
        rates[0] = joint_rate(e, integral, c, Kp, Ki, ff)
        rates[1:] = 1.0 * (theta0[1:] - theta[1:])   # the other five: held where they started
        peak = max(peak, abs(rates[0]))
        arm.command_velocities(rates)
        scene.step()
        t.append(k * dt); err.append(e)
    arm.command_velocities(np.zeros(6)); scene.step(); scene.release()
    return np.array(t), np.array(err), peak


def settling(t, e, frac=0.02):
    # The first time after which |e| stays within frac of its initial value (None if never).
    inside = np.abs(e) <= frac * abs(e[0])
    return next((t[i] for i in range(len(t)) if inside[i:].all()), None)

### A2. Four controllers, predicted first

🔮 **PREDICT, with numbers.** For each row, the Checker writes the final error and
how long the error takes to settle (to 2% of the initial 0.2 rad), using MR
11.3.1: the time constant of first-order error dynamics, and what a P controller
does when the reference moves at constant speed.

| run | `Kp` | `ff` | final error (rad) | 2% settling (s) |
|---|---|---|---|---|
| feedforward only | 0 | on | | |
| P only | 2 | off | | |
| P only | 5 | off | | |
| feedforward + P | 2 | on | | |
| feedforward + P | 5 | on | | |

In [ ]:
runs = [("feedforward only", dict(Kp=0, ff=True)),
        ("P only, Kp=2", dict(Kp=2, ff=False)),
        ("P only, Kp=5", dict(Kp=5, ff=False)),
        ("ff + P, Kp=2", dict(Kp=2, ff=True)),
        ("ff + P, Kp=5", dict(Kp=5, ff=True))]

fig, ax = plt.subplots(figsize=(7, 3.5))
for label, kw in runs:
    t, e, peak = track_joint(**kw)
    print(f"{label:>17}: final error {e[-1]:+.4f} rad, 2% settling {settling(t, e)} s")
    ax.plot(t, e, label=label)
ax.set_xlabel("t (s)"); ax.set_ylabel("error (rad)"); ax.axhline(0, color="gray", lw=0.5); ax.legend()

### A3. Your gain card

Your group's card gives a gain `Kp`, with feedforward on. The controller does
not act continuously: it reads the error, and the rate it sends is held for a
whole step of $\Delta t = 0.05$ s.

🔮 **PREDICT.** On the board, write the error after one step in terms of the error
now: with feedforward on, $\theta_e[k+1] = (\;?\;)\,\theta_e[k]$. What does that
factor say about your card's gain: does the error shrink smoothly, in one step,
by alternating sign, or not at all? Sketch the error for the first five steps.

In [ ]:
Kp_card = 5          # TODO: your card's gain

t, e, peak = track_joint(Kp=Kp_card, ff=True)
print("first six errors (rad):", np.round(e[:6], 4))
print("step-to-step ratios:   ", np.round(e[1:6] / e[:5], 3))
print(f"2% settling: {settling(t, e)} s;  largest commanded rate: {peak:.2f} rad/s")
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t, e, ".-"); ax.set_xlabel("t (s)"); ax.set_ylabel("error (rad)")
ax.set_title(f"Kp = {Kp_card}, Kp dt = {Kp_card * dt:.2f}"); ax.axhline(0, color="gray", lw=0.5)

📺 **Screens up** when the instructor calls it, plots facing the room, in order of
the cards' gains. Where does the step-to-step factor predict the change from
smooth to alternating, and from alternating to unstable? Did every robot do what
its factor said? Where it did not, look at the largest commanded rate.

### A4. If you are ahead: the integral term

MR's PI controller (eq. 11.13, no feedforward) removes the lag a P controller
leaves behind a constant-speed reference. Choose $K_i$ for critical damping (MR
11.3.1: $\omega_n = \sqrt{K_i}$, $\zeta = K_p / 2\sqrt{K_i}$).

🔮 **PREDICT.** With `Kp=5` and that `Ki`, does the lag go away? Now turn
feedforward on as well, with the same gains: better or worse than feedforward + P,
and why?

In [ ]:
Ki_crit = 0.0        # TODO: Ki for critical damping with Kp = 5

fig, ax = plt.subplots(figsize=(7, 3))
for label, kw in [("P only, Kp=5", dict(Kp=5, ff=False)),
                  ("PI, critically damped", dict(Kp=5, Ki=Ki_crit, ff=False)),
                  ("ff + P, Kp=5", dict(Kp=5, ff=True)),
                  ("ff + PI, same gains", dict(Kp=5, Ki=Ki_crit, ff=True))]:
    t, e, peak = track_joint(**kw)
    print(f"{label:>22}: final error {e[-1]:+.4f} rad, 2% settling {settling(t, e)} s, most negative {e.min():+.4f}")
    ax.plot(t, e, label=label)
ax.set_xlabel("t (s)"); ax.set_ylabel("error (rad)"); ax.axhline(0, color="gray", lw=0.5); ax.legend()

## Part B: The cup, in task space (MR 11.3.3)

Now the whole arm. The reference takes the cup from $X_A$ to $X_B$: 0.40 m along
a straight line while it turns 45° about the vertical, in 3 s with a quintic
time scaling (`screws.cartesian_trajectory`), then holds $X_B$ for a second.

MR's task-space law (eq. 11.16), in the cup's own frame:

$$\mathcal{V}_b = [\mathrm{Ad}_{X^{-1}X_d}]\,\mathcal{V}_d + K_p X_e + K_i \int_0^t X_e\,dt,
\qquad [X_e] = \log(X^{-1} X_d), \qquad \dot\theta = J_b^\dagger(\theta)\,\mathcal{V}_b .$$

$X$ is where the cup is, $X_d$ where it should be, and $\mathcal{V}_d$ the
reference's own twist, which `track` computes from consecutive reference poses.

### B1. Write the law

Fill in the three TODOs in `task_rates`. `screws.log6` and `screws.se3_to_vec`
give $X_e$; `screws.adjoint` and `screws.transform_inv` give the feedforward
term; `np.linalg.pinv` gives $J_b^\dagger$ (MR 6.3).

In [ ]:
def task_rates(model, theta, X, X_d, V_d, integral, Kp, Ki, ff):
    # MR eq. 11.16 and 6.3: commanded joint rates, and the configuration error X_e.
    X_err = np.zeros(6)      # TODO: the 6-vector X_e, from [X_e] = log(X^-1 X_d)
    V_ff = np.zeros(6)       # TODO: [Ad_{X^-1 X_d}] V_d if ff, else zero
    V = V_ff + Kp * X_err + Ki * integral
    J_b = model.jacobian_body(theta)
    theta_dot = np.zeros(6)  # TODO: J_b^+ V
    return theta_dot, X_err


OFFSET = np.array([0.35, -0.25, 0.3, 0.3, -0.3, 0.4])   # rad: the start, away from the reference


def track(Kp, Ki=0.0, ff=True, model=robot, feedback="model", vmax=np.pi, seconds=3.0, hold=1.0):
    # Track the line from X_A to X_B under task_rates, starting at theta_A + OFFSET.
    # feedback="model": X is the model's FK of the measured angles; "scene": X is cell.tool_frame().
    # vmax: if any commanded joint rate is above it, all six are scaled down together (None: no limit).
    start_at(theta_A + OFFSET)
    N = int(round(seconds / dt)) + 1
    ref = screws.cartesian_trajectory(X_A, X_B, seconds, N, "quintic") + [X_B] * int(round(hold / dt))
    arm.command_velocities(np.zeros(6)); scene.step()
    integral = np.zeros(6)
    log = dict(t=[], mm=[], deg=[], says_mm=[], peak=0.0, delivered=0.0)
    theta_prev = arm.theta()
    for k, X_d in enumerate(ref):
        X_next = ref[min(k + 1, len(ref) - 1)]
        V_d = screws.se3_to_vec(screws.log6(screws.transform_inv(X_d) @ X_next)) / dt
        theta = arm.theta()
        log["delivered"] = max(log["delivered"], np.abs(theta - theta_prev).max() / dt)   # the rate the arm really ran
        theta_prev = theta
        X = model.fk(theta) if feedback == "model" else cell.tool_frame()
        theta_dot, X_err = task_rates(model, theta, X, X_d, V_d, integral, Kp, Ki, ff)
        integral = integral + X_err * dt
        log["peak"] = max(log["peak"], np.abs(theta_dot).max())
        if vmax is not None and np.abs(theta_dot).max() > vmax:
            theta_dot = theta_dot * vmax / np.abs(theta_dot).max()
        # the error, from the simulator's own cup, at the same instant as X_d
        X_true = cell.tool_frame()
        e_true = screws.se3_to_vec(screws.log6(screws.transform_inv(X_true) @ X_d))
        log["t"].append(k * dt)
        log["mm"].append(1000 * np.linalg.norm(X_true[:3, 3] - X_d[:3, 3]))
        log["deg"].append(np.degrees(np.linalg.norm(e_true[:3])))
        log["says_mm"].append(1000 * np.linalg.norm(model.fk(theta)[:3, 3] - X_d[:3, 3]))
        arm.command_velocities(theta_dot)
        scene.step()
    arm.command_velocities(np.zeros(6)); scene.step(); scene.release()
    log = {key: (np.array(v) if isinstance(v, list) else v) for key, v in log.items()}
    ok = (log["mm"] < 5) & (log["deg"] < 1)
    log["converged"] = next((log["t"][i] for i in range(len(ok)) if ok[i:].all()), None)
    return log


def report(label, log):
    moving = (log["t"] >= 1.5) & (log["t"] <= 3.0)
    print(f"{label:>22}: converged {log['converged']} s; worst 1.5-3 s {log['mm'][moving].max():6.2f} mm; "
          f"end {log['mm'][-1]:.2f} mm, {log['deg'][-1]:.3f} deg; "
          f"rates asked {log['peak']:.2f}, ran {log['delivered']:.2f} rad/s")


def plot(logs):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
    for label, log in logs:
        a1.plot(log["t"], log["mm"], label=label); a2.plot(log["t"], log["deg"], label=label)
    a1.set_xlabel("t (s)"); a1.set_ylabel("position error (mm)")
    a2.set_xlabel("t (s)"); a2.set_ylabel("orientation error (deg)"); a2.legend()
    fig.tight_layout()

### B1. Three runs, predicted first

The arm starts about 0.2 m and 26° away from $X_A$.

🔮 **PREDICT.** For each run: does the cup ever reach the line? How long does it
take? What error is left while the cup moves? For *P only*, the cup's peak speed
along a quintic is $\tfrac{15}{8}\,L/T$; use MR 11.3.1's lag behind a moving
reference.

In [ ]:
logs = []
for label, kw in [("feedforward only", dict(Kp=0)),
                  ("ff + P, Kp=5", dict(Kp=5)),
                  ("P only, Kp=5", dict(Kp=5, ff=False))]:
    log = track(**kw)
    report(label, log)
    logs.append((label, log))
plot(logs)

### B2. Your group's experiment

Your card names one of the three. Predict on the board, run it, and be ready to
tell the room what you found at the exit check-in.

**(a) The rate limit.** `track` scales the joint rates down if any is above
$\pi$ rad/s, a real UR5's 180°/s. 🔮 With `Kp=20`, what is the largest rate the
controller *asks* for? Run it with `vmax=None` and with the limit. Does the
simulated arm deliver what was asked? Would the real one?

In [ ]:
for label, kw in [("Kp=20, limited", dict(Kp=20)), ("Kp=20, no limit", dict(Kp=20, vmax=None))]:
    report(label, track(**kw))

**(b) The integral term.** 🔮 Add `Ki=2` to feedforward + P with `Kp=5`. Does
the cup reach the line sooner or later? What is left at the end of the move?
Compare with A4: what is there for an integrator to remove here?

In [ ]:
logs = [(label, track(**kw)) for label, kw in [("ff + P, Kp=5", dict(Kp=5)),
                                                 ("ff + PI, Kp=5, Ki=2", dict(Kp=5, Ki=2))]]
for label, log in logs:
    report(label, log)
plot(logs)

**(c) A model that is wrong.** Model the cup 5 mm too short. 🔮 What does the
controller think its error is at the end, and what does the simulator say? Then
feed back the simulator's own cup, `feedback="scene"` (`cell.tool_frame()`, the
measured pose) instead of the model's forward kinematics. Which error goes away?

In [ ]:
M_short = M @ screws.rp_to_transform(np.eye(3), [0, 0, -0.005])    # the cup's face 5 mm nearer the flange
robot_short = screws.Robot.from_screw_axes(M_short, list(S.T), name="UR5 + short cup")

for label, kw in [("short model", dict(Kp=5, model=robot_short)),
                  ("short model, scene X", dict(Kp=5, model=robot_short, feedback="scene"))]:
    log = track(**kw)
    report(label, log)
    print(f"{'':>22}  the model's own error at the end: {log['says_mm'][-1]:.2f} mm")

## Part C: Converge

Wait for the instructor. Every group runs the same gain at the same moment, with
CoppeliaSim's window on screen, and we compare the robots side by side.

In [ ]:
Kp_converge = 5      # the instructor gives this in class

report(f"Kp={Kp_converge}", track(Kp=Kp_converge))

When you are done for the day:

In [ ]:
close(scene)